In [1]:
#Appel des librairies et du Dataframe
%config Completer.use_jedi = False
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.express as px

df_raw=pd.read_csv('000_df_RAW.csv', low_memory=False)

df_raw

,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,vosp,prof,plan,lartpc,larrout,surf,infra,situ,env1,vma
0,200500000001,1.0,1,4,1,1.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
1,200500000001,1.0,1,3,2,3.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
2,200500000001,2.0,2,1,1,0.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
3,200500000001,4.0,2,1,1,0.0,31.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
4,200500000001,5.0,2,1,1,0.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1,4,1,0.0,NaN,0.0,0,-1.0,...,0.0,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0
2509722,202100056516,1.0,1,4,1,5.0,NaN,0.0,0,-1.0,...,0.0,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0
2509723,202100056517,1.0,1,3,1,0.0,NaN,0.0,0,-1.0,...,0.0,1.0,2.0,NaN,-1,2.0,5.0,3.0,NaN,80.0
2509724,202100056518,1.0,1,3,1,3.0,NaN,-1.0,-1,-1.0,...,0.0,1.0,1.0,NaN,-1,2.0,0.0,1.0,NaN,70.0


In [2]:
#***On rend toutes les infos bizarres en NaN**
df=df_raw.copy(deep=True)
print(df.isna().sum().sum())

Nan_originaux=df.isna().sum()

#Variables de usagers___________________________________________________________________________________________________________________________________
df.place.replace([-1,0],np.NaN,inplace=True) #0 est non défini pour place
df.catu.replace(4,np.NaN,inplace=True) #4 est non défini pour catu
df.grav.replace(-1,np.NaN,inplace=True) #-1 est non défini pour grav
df.sexe.replace(-1,np.NaN,inplace=True) #-1 est non défini pour sexe
df.trajet.replace([-1,0],np.NaN,inplace=True) #-1 et 0 sont non défini
#secu sera géré plus tard
df.locp.replace([-1,9],0,inplace=True) #-1 est non renseigné et 0 est sans objet, on rassemble les deux
df.loc[df['catu'] == 3, 'locp'] = df.loc[df['catu'] == 3, 'locp'].replace(0, np.NaN)#Localisation du piéton filtrée pour les cas où catu==3 (concerne les piétons) : on NaN la ligne si non resneignée

df['actp']=df.actp.replace({'0.0':0,'-1':0,' -1':0,'0':0,'B':0,'A' : 99, '7':0 , '8':0 }).astype(float)#-1 est non renseigné et 0 est sans objet,B est inconnu, on rassemble tout. astype float permet de mélanger les formats x et x.0
df.loc[df['catu'] == 3, 'actp'] = df.loc[df['catu'] == 3, 'actp'].replace(0, np.NaN)#Localisation du piéton filtrée pour les cas où catu==3 (concerne les piétons) : on NaN la ligne si non resneignée

df['etatp']=df.etatp.replace({-1:0})#-1 est non renseigné et 0 est sans objet, on assemble
df.loc[df['catu'] == 3, 'etatp'] = df.loc[df['catu'] == 3, 'etatp'].replace(0, np.NaN)#Etat seul ou accompagné du piéton filtrée pour les cas où catu==3 (concerne les piétons) : NaN la ligne si non resneignée

temp=df['an'].copy(deep=True)
temp[temp<20]+=2000     
df['age']=temp-df['an_nais']                #Creation de la colonne age, et suppression de la colonne an_nais
df.drop(columns="an_nais",inplace=True)
med_age=df['age'].median()                   #Remplacement des NaN par l'age median
df['age'].fillna(med_age,inplace=True)

#id_vehicule à supprimer
#secu1/2/3 aussi

#Variables de vehi___________________________________________________________________________________________________________________________________
df.senc.replace([-1,0,3],np.NaN,inplace=True) #-1 est non défini pour place, 0 =inconnu et 3 est Absence d'info
df.catv.replace([-1,0],np.NaN,inplace=True) #-1 est non défini pour place, 0 =indéterminable
#occutc ne concerne que les transports en communs. On s'en occupera plus tard si vraiment on veut s'en servir, mais je pense qu'on va juste retirer la colonne.
#obs et obsm se completent surement, on vérifiera plus tard
df.choc.replace(-1,np.NaN,inplace=True) #-1 = non renseigné. On garde 0 car il représente "aucun choc", qui peut arriver dans certains accident moteurs par exemple
df.manv.replace([-1,0],np.NaN,inplace=True) #-1 Non renseigné. 0 = Inconnu
#id vehicule sert dans al clé de merge puis est supprimée
#motor n'est présent que depuis 2019, pas assez de donnée, on supprimera la colonne, si on veut travailler seulement sur 2019-2021 dans un second temps, on recommencera le préprocessing de toutes façons.



#Variables de lieux___________________________________________________________________________________________________________________________________

#Ok pour catr
#voie sera supprimée car elle a 32000 modalités différentes et est sans interet
#v1 sera supprimée car elle a 50% de nan , et la quasi totalité de ce qu'il reste est 0 ou -1 et est sans intéret
#v2 sera supprimée, un savant mélange des problèmes de voie et v1
df.circ.replace([-1,0],np.NaN,inplace=True) # -1 Non renseigné, 0 n'a pas d'explication dans les documents, et pas de sens puisque les autres modalités possibles couvrent tous les cas
df.nbv.replace([-1,0],np.NaN,inplace=True) # -1 = non resneigné je suppose. Est-ce que 0 a du sens pour un nombre de voies ? Y a-t-il des accidents dans des lieux sans voies ? Parking ? Même un parking possède des sens de circulation. Pas clair, on retire.
df.vosp.replace(-1,np.NaN,inplace=True) # -1 = non resneigné 
#pr représente la borne kilométrique la plus proche. inutile puisqu'on ne peut pas utiliser les noms de voies
#pr1 représente la distance à la borne la plus proche, inutile puisqu'on ne peut pas utiliser les noms de voies
df.prof.replace([-1,0],np.NaN,inplace=True) # Inclinaison de la route : -1 = non resneigné, 0 n'est pas expliqué dans les documents. On met NaN, mais je pense que le fillna pourra prendre 1=Plat
df.plan.replace([-1,0],np.NaN,inplace=True) # courbure de la route : -1 = non resneigné, 0 n'est pas expliqué dans les documents. On met NaN, mais je pense que le fillna pourra prendre 1=rectiligne
#lartpc : la ligne lieux.lartpc.str.replace(',','.').astype(float).describe() permet de voir la répartition de la variable : 75% valent 0. Je pense qu'on supprimera la colonne.
#larrout : largeur de la route possède une trop grosse part de 0 et -1. A supprimer je pense
df.surf.replace([-1,0],np.NaN,inplace=True)#Etat de surface : 0 et -1 sont non renseigné
df.infra.replace(-1,np.NaN,inplace=True) #Amenagement : -1 non renseigné. On garde les 0 = Aucun
df.situ.replace([-1,0],np.NaN,inplace=True) #Situation de l'accident : -1 non resneigné, 0 = aucun (n'a pas de sens)
#env1 : 15% de Nan, 3 modalités, dont la moitié pour 0. Je propose de jeter si on ne sait pas ce que c'est. (aucune explication dans les documents)
#vma n'existe que depuis 2019



#Variables de carac___________________________________________________________________________________________________________________________________

#an ok
#mois ok
#jour ok
#hrmn ok
df.lum.replace(-1,np.NaN,inplace=True) #-1 est non défini
#agg ok
df.int.replace([-1,0],np.NaN,inplace=True) #-1 et 0 sont non défini
df.atm.replace(-1,np.NaN,inplace=True) #-1 est non défini
df.col.replace(-1,np.NaN,inplace=True) #-1 est non renseigné
#adr sera supprimée car inexploitable : 467491 modalités différentes
#gps sera traité plus tard
#lat sera traité plus tard
#long sera traité plus tard
#dep peut être abandonné ou traité plus tard
#com possède 20k modalités différentes, on supprimera



Nan_réels=df.isna().sum()

df_temp1=df
print(df.isna().sum().sum())
df

25092019
28806102


,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,prof,plan,lartpc,larrout,surf,infra,situ,env1,vma,age
0,200500000001,1.0,1.0,4.0,1.0,1.0,11.0,0.0,0.0,0.0,...,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN,29.0
1,200500000001,1.0,1.0,3.0,2.0,3.0,11.0,0.0,0.0,0.0,...,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN,37.0
2,200500000001,2.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN,41.0
3,200500000001,4.0,2.0,1.0,1.0,NaN,31.0,0.0,0.0,0.0,...,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN,1.0
4,200500000001,5.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,NaN,0.0,0.0,0.0,...,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0,46.0
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,NaN,0.0,0.0,0.0,...,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0,19.0
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,NaN,0.0,0.0,0.0,...,1.0,2.0,NaN,-1,2.0,5.0,3.0,NaN,80.0,22.0
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,NaN,0.0,0.0,0.0,...,1.0,1.0,NaN,-1,2.0,0.0,1.0,NaN,70.0,53.0


In [3]:
#**On retire les colonnes avec trop de NaN et aucune solution pour les remplacer, où les colonnes inutilisables****
df=df_temp1.copy(deep=True)
print(df.isna().sum().sum())

#infos venant de 'vehicule' : ces colonnes n'existent qu'a partir de 2019
df=df.drop(columns=['id_vehicule','motor','occutc'])

#infos venant de 'lieux' : 
df=df.drop(columns=['voie','v1','v2','lartpc','larrout','env1','vma','pr','pr1','nbv'])

#Variables de carac_____________________________
df=df.drop(columns=['adr',"com"])



df_temp2=df
print(df.isna().sum().sum())
df

28806102
14209346


,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,dep,catr,circ,vosp,prof,plan,surf,infra,situ,age
0,200500000001,1.0,1.0,4.0,1.0,1.0,11.0,0.0,0.0,0.0,...,590,3.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0,29.0
1,200500000001,1.0,1.0,3.0,2.0,3.0,11.0,0.0,0.0,0.0,...,590,3.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0,37.0
2,200500000001,2.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,590,3.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0,41.0
3,200500000001,4.0,2.0,1.0,1.0,NaN,31.0,0.0,0.0,0.0,...,590,3.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
4,200500000001,5.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,590,3.0,2.0,0.0,1.0,1.0,1.0,0.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,NaN,0.0,0.0,0.0,...,26,3.0,2.0,0.0,2.0,2.0,2.0,0.0,1.0,46.0
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,NaN,0.0,0.0,0.0,...,26,3.0,2.0,0.0,2.0,2.0,2.0,0.0,1.0,19.0
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,NaN,0.0,0.0,0.0,...,33,4.0,2.0,0.0,1.0,2.0,2.0,5.0,3.0,22.0
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,NaN,0.0,0.0,0.0,...,78,3.0,2.0,0.0,1.0,1.0,2.0,0.0,1.0,53.0


In [4]:
#Melange des infos sur secu et secu1/2/3 puis dummification pour retirer les colonnes secu
df=df_temp2.copy(deep=True)
print(df.isna().sum().sum())


#uniformisation des données sur "secu" avec leur encodage sur 'secu1'
temp=df.secu.copy(deep=True)
temp[temp<=10]=0
temp[temp%10==3]=0
temp[temp%10==2]=0
temp[temp%10==0]=0
temp=temp//10


df.secu1=df.secu1.replace(np.NaN,0)
temp=temp.replace(np.NaN,0)

df["secu1"]=df.secu1+temp

dummies=df[['secu1','secu2','secu3']]

#On ne garde que certains éléments, les autres sont extremement minoritaires, entre autre car jusqu'a 2018, seuls ces 4 là sont renseignés :
dummies=dummies.replace({1:'Ceinture',2:'Casque',3:'Dispositif_Enfant',4:'Equipement_Réfléchissant',5:np.NaN,6:np.NaN,7:np.NaN,0:np.NaN,8:np.NaN,-1:np.NaN,9:np.NaN})

dummies = pd.get_dummies(dummies)

df['Casque']=dummies.secu1_Casque+dummies.secu2_Casque+dummies.secu3_Casque
df['Ceinture']=dummies.secu1_Ceinture+dummies.secu2_Ceinture+dummies.secu3_Ceinture
df['Dispositif_Enfant']=dummies.secu1_Dispositif_Enfant+dummies.secu2_Dispositif_Enfant+dummies.secu3_Dispositif_Enfant
df['Equipement_Réfléchissant']=dummies.secu1_Equipement_Réfléchissant+dummies.secu2_Equipement_Réfléchissant+dummies.secu3_Equipement_Réfléchissant

df=df.drop(columns=['secu','secu1','secu2','secu3']) #on retire les colonnes devenues inutiles
df_temp3=df
print(df.isna().sum().sum())
df

14209346
7358737


,Num_Acc,place,catu,grav,sexe,trajet,locp,actp,etatp,num_veh,...,prof,plan,surf,infra,situ,age,Casque,Ceinture,Dispositif_Enfant,Equipement_Réfléchissant
0,200500000001,1.0,1.0,4.0,1.0,1.0,0.0,0.0,0.0,A01,...,1.0,1.0,1.0,0.0,1.0,29.0,False,True,False,False
1,200500000001,1.0,1.0,3.0,2.0,3.0,0.0,0.0,0.0,B02,...,1.0,1.0,1.0,0.0,1.0,37.0,False,True,False,False
2,200500000001,2.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,B02,...,1.0,1.0,1.0,0.0,1.0,41.0,False,True,False,False
3,200500000001,4.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,B02,...,1.0,1.0,1.0,0.0,1.0,1.0,False,False,True,False
4,200500000001,5.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,B02,...,1.0,1.0,1.0,0.0,1.0,7.0,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,0.0,0.0,0.0,A01,...,2.0,2.0,2.0,0.0,1.0,46.0,False,True,False,False
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,0.0,0.0,0.0,B01,...,2.0,2.0,2.0,0.0,1.0,19.0,False,True,False,False
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,0.0,0.0,0.0,A01,...,1.0,2.0,2.0,5.0,3.0,22.0,False,False,False,False
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,0.0,0.0,0.0,A01,...,1.0,1.0,2.0,0.0,1.0,53.0,False,True,False,False


In [5]:
#date column in the datetime format
df=df_temp3.copy(deep=True)
print(df.isna().sum().sum())

df.an.replace({5:2005,6:2006,7:2007,8:2008,9:2009},inplace=True) #fix an format
df.an.replace({10:2010,11:2011,12:2012,13:2013,14:2014,15:2015,16:2016,17:2017,18:2018},inplace=True) #fix an format
df['date'] = df['an'].astype(str) + '-' + df['mois'].astype(str) + '-' + df['jour'].astype(str)
df['date'] = pd.to_datetime(df['date'], yearfirst = True)
df['hrmn2']=df['hrmn'].astype(str) #fix hrmn format
df['hrmn2']=df['hrmn2'].apply(lambda x: x.replace(':', '') if ':' in x else x) #fix hrmn format
df['hrmn2'] = df['hrmn2'].apply(lambda x: str(x).zfill(4))  # corrige les heures à minuit dont les 0 de gauche n'ont pas été rentrés
df['hrmn2'] = pd.to_datetime(df['hrmn2'], format='%H%M', errors='raise')  #fix hrmn format
df['date'] = df['date'] + pd.to_timedelta(df['hrmn2'].dt.hour, unit='h') + pd.to_timedelta(df['hrmn2'].dt.minute, unit='m') #adding hour+time
df=df.drop(['hrmn2','hrmn','an','mois','jour'],axis=1) #removing dummy columns


df_temp4=df
print(df.isna().sum().sum())
df

7358737
7358737


,Num_Acc,place,catu,grav,sexe,trajet,locp,actp,etatp,num_veh,...,plan,surf,infra,situ,age,Casque,Ceinture,Dispositif_Enfant,Equipement_Réfléchissant,date
0,200500000001,1.0,1.0,4.0,1.0,1.0,0.0,0.0,0.0,A01,...,1.0,1.0,0.0,1.0,29.0,False,True,False,False,2005-01-12 19:00:00
1,200500000001,1.0,1.0,3.0,2.0,3.0,0.0,0.0,0.0,B02,...,1.0,1.0,0.0,1.0,37.0,False,True,False,False,2005-01-12 19:00:00
2,200500000001,2.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,B02,...,1.0,1.0,0.0,1.0,41.0,False,True,False,False,2005-01-12 19:00:00
3,200500000001,4.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,B02,...,1.0,1.0,0.0,1.0,1.0,False,False,True,False,2005-01-12 19:00:00
4,200500000001,5.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,B02,...,1.0,1.0,0.0,1.0,7.0,False,True,False,False,2005-01-12 19:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,0.0,0.0,0.0,A01,...,2.0,2.0,0.0,1.0,46.0,False,True,False,False,2021-01-01 18:00:00
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,0.0,0.0,0.0,B01,...,2.0,2.0,0.0,1.0,19.0,False,True,False,False,2021-01-01 18:00:00
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,0.0,0.0,0.0,A01,...,2.0,2.0,5.0,3.0,22.0,False,False,False,False,2021-01-01 10:55:00
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,0.0,0.0,0.0,A01,...,1.0,2.0,0.0,1.0,53.0,False,True,False,False,2021-01-02 18:00:00


In [6]:
"""Dumification des variables"""

df=df_temp4.copy(deep=True)
print(df.isna().sum().sum())

#Dummification de place___________________________________________________________________________________________________________________________________

df['place']=df['place'].replace({1:'Avant_Gauche',2:'Avant_Droit',3:'Arriere_Droit',4:'Arriere_Gauche',5:"Autre",6:"Autre",7:"Autre",8:"Autre",9:"Autre",10:np.NaN})    #on met toutes les places "du milieux" dans autres car elles sont trop peu présentes sinon
                                                                                                                                                                        #de plus, on retire les piéton (10) car l'info sera plus précise dans "catu"
df = pd.concat([df.drop(columns='place'), pd.get_dummies(df['place'],prefix='place')], axis=1)

print(df.isna().sum().sum())


#Dummification de catu___________________________________________________________________________________________________________________________________
df['catu']=df['catu'].replace({1:'Conducteur',2:'Passager',3:'Pieton'})  
df = pd.concat([df.drop(columns='catu'), pd.get_dummies(df['catu'],prefix='catu')], axis=1)

print(df.isna().sum().sum())

#Dummification de sexe___________________________________________________________________________________________________________________________________
df['sexe']=df['sexe'].replace({1:'Masculin',2:'Feminin'})  
df = pd.concat([df.drop(columns='sexe'), pd.get_dummies(df['sexe'],prefix='sexe')], axis=1)

print(df.isna().sum().sum())

#Dummification de trajet___________________________________________________________________________________________________________________________________
df['trajet']=df['trajet'].replace({1:'Domicile_Travail',2:'Domicile_Travail',3:'Achats',4:'Utilisation_Professionnelle',5:"Loisirs",9:"Autre"}) 
df = pd.concat([df.drop(columns='trajet'), pd.get_dummies(df['trajet'],prefix='trajet')], axis=1)

print(df.isna().sum().sum())

#Dummification de locp___________________________________________________________________________________________________________________________________
df['locp']=df['locp'].replace({0:np.NaN,1:'Chausse_Hors_Passage_Pieton',2:'Chausse_Hors_Passage_Pieton',3:'Sur_Passage_Pieton',4:'Sur_Passage_Pieton',5:"Trottoir/Bord_de_route",6:"Trottoir/Bord_de_route",7:"Trottoir/Bord_de_route",8:"Trottoir/Bord_de_route"}) 
df = pd.concat([df.drop(columns='locp'), pd.get_dummies(df['locp'],prefix='locp')], axis=1)

print(df.isna().sum().sum())


#Dummification de actp___________________________________________________________________________________________________________________________________
df['actp']=df['actp'].replace({0:np.NaN,1:'Deplacement_sens_vehicule',2:'Deplacement_sens_inv_vehicule',3:'Traversant',4:'Autre',5:"Jouant_Courant",6:"Avec_Animal",9:'Autre',99:"Autre"}) 
df = pd.concat([df.drop(columns='actp'), pd.get_dummies(df['actp'],prefix='actp')], axis=1)

print(df.isna().sum().sum())


#Dummification de etatp___________________________________________________________________________________________________________________________________
df['etatp']=df['etatp'].replace({0:np.NaN,1:'Seul',2:'Groupe',3:'Groupe'}) 
df = pd.concat([df.drop(columns='etatp'), pd.get_dummies(df['etatp'],prefix='etatp')], axis=1)

print(df.isna().sum().sum())


#Dummification de senc___________________________________________________________________________________________________________________________________
df['senc']=df['senc'].replace({0:np.NaN,1:'Repere_Croissant',2:'Repere_Decroissant'}) 
df = pd.concat([df.drop(columns='senc'), pd.get_dummies(df['senc'],prefix='senc')], axis=1)

print(df.isna().sum().sum())


#Dummification de prof___________________________________________________________________________________________________________________________________

df['prof']=df['prof'].replace({1:np.NaN,2:'Pente',3:'Pente',4:'Pente'})     #.fillna('Plat')                      #possibilité de séparer pente suivant l'endroit pente/sommet/bas, mais je préfère limiter le nombre de dummy
                                                                                                                #De plus on met Plat en NaN, car 2 dummies complémentaires ne donnent pas plus d'information qu'un dummy Booléen
df = pd.concat([df.drop(columns='prof'), pd.get_dummies(df['prof'],prefix='prof')], axis=1)

print(df.isna().sum().sum())


#Dummification de plan___________________________________________________________________________________________________________________________________

df['plan']=df['plan'].replace({1:np.NaN,2:'Courbe',3:'Courbe',4:'Courbe'})    #.fillna('Rectiligne')       #pareil qu'au dessus
df = pd.concat([df.drop(columns='plan'), pd.get_dummies(df['plan'],prefix='plan')], axis=1)

print(df.isna().sum().sum())


#Dummification de manv___________________________________________________________________________________________________________________________________
dico={  1:'Aucune_Manoeuvre',2:'Aucune_Manoeuvre',3:"Depasse",4:'Arret/se_gare',5:'Contresens',6:"Autre",7:"Autre",8:"Autre",9:"Deport",
        10:"Tourne",11:"Deport",12:"Deport",13:"Deport",14:"Deport",15:"Tourne",16:"Tourne",17:"Depasse",18:"Depasse",19:"Autre",
        20:"Arret/se_gare",21:"Deport",22:"Arret/se_gare",23:"Arret/se_gare",24:"Arret/se_gare",25:"Autre",26:"Autre"}                                        #pleins de dico possible. J'ai essayé de limiter les dummy, sans perdre trop d'infos
df['manv']=df['manv'].replace(dico)
df = pd.concat([df.drop(columns='manv'), pd.get_dummies(df['manv'],prefix='manv')], axis=1)

print(df.isna().sum().sum())


#Dummification de situ___________________________________________________________________________________________________________________________________

df['situ']=df['situ'].replace({1:'Chausse',2:'Trottoir/Bord_de_route',3:'Trottoir/Bord_de_route',4:'Trottoir/Bord_de_route',5:'Trottoir/Bord_de_route',6:'Trottoir/Bord_de_route',8:np.NaN})    # Les 2 dummies ne sont pas complémentaires
df = pd.concat([df.drop(columns='situ'), pd.get_dummies(df['situ'],prefix='situ')], axis=1)

print(df.isna().sum().sum())


#Dummification de circ___________________________________________________________________________________________________________________________________

df['circ']=df['circ'].replace({1:'Sens_Unique',2:'Bidirectionnelle',3:'Chausse_Separee',4:'Chausse_Separee'})    #Les voix d'affectations (4) sont gloablement sur autoroutes et voix rapides, donc à chaussées séparées (3)
df = pd.concat([df.drop(columns='circ'), pd.get_dummies(df['circ'],prefix='circ')], axis=1)

print(df.isna().sum().sum())


#Dummification de surf___________________________________________________________________________________________________________________________________

df['surf']=df['surf'].replace({1:np.NaN,2:'Glissant',3:'Glissant',4:'Glissant',5:'Glissant',6:'Glissant',7:'Glissant',8:'Glissant',9:'Glissant'})    # On donne l'information "normal" via l'absence de dummy positif pour "glissant" qui synthetise les différents états d'humidité/neige/huile/verglas/...
df = pd.concat([df.drop(columns='surf'), pd.get_dummies(df['surf'],prefix='surf')], axis=1)

print(df.isna().sum().sum())


#Dummification de infra___________________________________________________________________________________________________________________________________

df['infra']=df['infra'].replace({0:np.NaN,1:"Autre",2:'Autre',3:'Autre',4:'Autre',5:'Carrefour',6:'Autre',7:'Autre',8:'Autre',9:'Autre'})    #On garde les infos Carrefour, Autre, et Aucun (=False aux 2 autres)
df = pd.concat([df.drop(columns='infra'), pd.get_dummies(df['infra'],prefix='infra')], axis=1)

print(df.isna().sum().sum())


#Dummification de vosp___________________________________________________________________________________________________________________________________

df['vosp']=df['vosp'].replace({0:np.NaN,1:"Cycl/Reserv",2:'Cycl/Reserv',3:'Cycl/Reserv'})           #On garde l'info la présense d'un affichage sur un dummy, indépendament du type d'affichage (90% des lignes sont à 0)
df = pd.concat([df.drop(columns='vosp'), pd.get_dummies(df['vosp'],prefix='vosp')], axis=1)


df_temp5=df
print(df.isna().sum().sum())
df


7358737
7174831
7171271
7168209
6461674
6385137
6318175
6250924
4365595
4216928
4070317
3882058
3770301
3642437
3576759
3561118
3552247


,Num_Acc,grav,num_veh,catv,obs,obsm,choc,lum,agg,int,...,manv_Tourne,situ_Chausse,situ_Trottoir/Bord_de_route,circ_Bidirectionnelle,circ_Chausse_Separee,circ_Sens_Unique,surf_Glissant,infra_Autre,infra_Carrefour,vosp_Cycl/Reserv
0,200500000001,4.0,A01,7.0,0.0,2.0,1.0,3.0,2,1.0,...,False,True,False,True,False,False,False,False,False,False
1,200500000001,3.0,B02,7.0,0.0,2.0,8.0,3.0,2,1.0,...,True,True,False,True,False,False,False,False,False,False
2,200500000001,1.0,B02,7.0,0.0,2.0,8.0,3.0,2,1.0,...,True,True,False,True,False,False,False,False,False,False
3,200500000001,1.0,B02,7.0,0.0,2.0,8.0,3.0,2,1.0,...,True,True,False,True,False,False,False,False,False,False
4,200500000001,1.0,B02,7.0,0.0,2.0,8.0,3.0,2,1.0,...,True,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,4.0,A01,7.0,0.0,2.0,1.0,3.0,1,1.0,...,False,True,False,True,False,False,True,False,False,False
2509722,202100056516,4.0,B01,7.0,0.0,2.0,3.0,3.0,1,1.0,...,False,True,False,True,False,False,True,False,False,False
2509723,202100056517,3.0,A01,7.0,6.0,0.0,5.0,1.0,1,2.0,...,False,False,True,True,False,False,True,False,True,False
2509724,202100056518,3.0,A01,7.0,0.0,2.0,1.0,3.0,1,1.0,...,False,True,False,True,False,False,True,False,False,False


In [7]:
#Suite
df=df_temp5.copy(deep=True)
print(df.isna().sum().sum())


#Dummification de col___________________________________________________________________________________________________________________________________

df['col']=df['col'].replace({1:"2_Vehicules",2:'2_Vehicules',3:'2_Vehicules',4:'3_Vehicules',5:'3_Vehicules',6:np.NaN,7:np.NaN})
df = pd.concat([df.drop(columns='col'), pd.get_dummies(df['col'],prefix='col')], axis=1)

print(df.isna().sum().sum())


#Dummification de obs et obsm___________________________________________________________________________________________________________________________________
df['obs'].replace(-1,0,inplace=True)
df['obs'] = np.where(df['obs'] != 0, 99, df['obsm'])    #obs prend la valeur de obsm quand obs vaut 0, et 99 sinon
df['obs']=df['obs'].replace({1:"Piéton",2:'Vehicule',4:'Vehicule',5:np.NaN,6:np.NaN,9:np.NaN,0:np.NaN,99:"Fixe",-1:np.NaN})
df = pd.concat([df.drop(columns=['obs','obsm']), pd.get_dummies(df['obs'],prefix='obs')], axis=1)
print(df.isna().sum().sum())


#Dummification de catv___________________________________________________________________________________________________________________________________
dico={  1:'2_roues',2:'2_roues',3:"Voiture",4:'2_roues',5:'2_roues',6:"2_roues",7:"Voiture",8:"Voiture",9:"Voiture",
        10:"Utilitaire",11:"Utilitaire",12:"Utilitaire",13:"Lourd",14:"Lourd",15:"Lourd",16:"Lourd",17:"Lourd",18:"Lourd",19:"Lourd",
        20:"Lourd",21:"Lourd",30:"2_roues",31:"2_roues",32:"2_roues", 33:"2_roues",34:"2_roues",35:"2_roues",36:"2_roues",37:"Lourd",38:"Lourd",39:"Lourd",
        40:"Lourd",41:"2_roues",42:"2_roues",43:"2_roues",50:np.NaN,60:np.NaN,80:np.NaN,99:np.NaN}                                        
        #pleins de dico possible. J'ai essayé de limiter les dummy, sans perdre trop d'infos. Quad en 2 roues car vehicule sans habitacle et port du casque obligatoire. 
        # Vélos et trottinettes elec en np.NaN car pas vraiment un véhicule et trop peu nombreux pour un dummy

df['catv']=df['catv'].replace(dico)
df = pd.concat([df.drop(columns='catv'), pd.get_dummies(df['catv'],prefix='catv')], axis=1)
print(df.isna().sum().sum())


#Dummification de choc___________________________________________________________________________________________________________________________________

df['choc']=df['choc'].replace({0:np.NaN,1:"Avant",2:'Droit',3:'Gauche',4:'Arriere',5:'Arriere',6:'Arriere',7:'Droit',8:'Gauche',9:'Tonneaux'})  #Pour equilibrer les catégories, Av_D et Av_G sont dans Droite et Gauche et Tous les Arrieres sont ensemble
df = pd.concat([df.drop(columns='choc'), pd.get_dummies(df['choc'],prefix='choc')], axis=1)
print(df.isna().sum().sum())


#Dummification de int___________________________________________________________________________________________________________________________________

df['int']=df['int'].replace({1:np.NaN,2:'X',3:'TouY',4:'TouY',5:'Autre',6:'Autre',7:'Autre',8:'Autre',9:'Autre'})  # hors inter est en np.NaN, inter T et Y ensemble
df = pd.concat([df.drop(columns='int'), pd.get_dummies(df['int'],prefix='int')], axis=1)
print(df.isna().sum().sum())


#Dummification de atm___________________________________________________________________________________________________________________________________

df['atm']=df['atm'].replace({1:np.NaN,2:'Faible',3:'Eleve',4:'Eleve',5:'Eleve',6:'Eleve',7:'Faible',8:'Faible',9:'Eleve'})  
# normal en np.NaN, couvert/pluie legere/eblouissant = Faible , pluieforte/tempete/neige/brouillard = Eleve. Autre en Eleve parce que je suppose que ce qui n'est pas déja listé est un évenement extreme (inondation ? Orage/tonnere ?)
df = pd.concat([df.drop(columns='atm'), pd.get_dummies(df['atm'],prefix='atm')], axis=1)
print(df.isna().sum().sum())


#Correction de lum___________________________________________________________________________________________________________________________________

df['lum']=df['lum'].replace({1:False,2:True,3:True,4:True,5:True})  #Booléen de "Durant la Nuit ?"


#Correction de agg___________________________________________________________________________________________________________________________________

df['agg']=df['agg'].replace({1:False,2:True})                    #Booléen de "En Agglomération ?"


#Dummification de catr___________________________________________________________________________________________________________________________________

df['catr']=df['catr'].replace({1:"Nationnale/Autoroute",2:'Nationnale/Autoroute',3:'Departementale',4:'Voix_Communale',5:np.NaN,6:np.NaN,7:np.NaN,8:np.NaN,9:np.NaN})  #Suivant le Value_count : 4=1169935 3=826731 1=229036 2=213233, les autres sont -50k, Nan vaut Autre
df = pd.concat([df.drop(columns='catr'), pd.get_dummies(df['catr'],prefix='catr')], axis=1)


#Dummification de dep___________________________________________________________________________________________________________________________________ 
#_Merci ChatGPT pour le dictionnaire
departement_to_region = {
    '1': 'Auvergne-Rhône-Alpes',    '2': 'Hauts-de-France',    '3': 'Auvergne-Rhône-Alpes',    '4': 'Provence-Alpes-Côte d\'Azur',    '5': 'Provence-Alpes-Côte d\'Azur',    '6': 'Provence-Alpes-Côte d\'Azur',    '7': 'Auvergne-Rhône-Alpes',    '8': 'Grand Est',
    '9': 'Occitanie',        '01': 'Auvergne-Rhône-Alpes',    '02': 'Hauts-de-France',    '03': 'Auvergne-Rhône-Alpes',    '04': 'Provence-Alpes-Côte d\'Azur',    '05': 'Provence-Alpes-Côte d\'Azur',    '06': 'Provence-Alpes-Côte d\'Azur',    '07': 'Auvergne-Rhône-Alpes',
    '08': 'Grand Est',    '09': 'Occitanie',    '10': 'Grand Est',    '11': 'Occitanie',    '12': 'Occitanie',    '13': 'Provence-Alpes-Côte d\'Azur',    '14': 'Normandie',    '15': 'Auvergne-Rhône-Alpes',    '16': 'Nouvelle-Aquitaine',    '17': 'Nouvelle-Aquitaine',
    '18': 'Centre-Val de Loire',    '19': 'Nouvelle-Aquitaine',    '21': 'Bourgogne-Franche-Comté',    '22': 'Bretagne',    '23': 'Nouvelle-Aquitaine',    '24': 'Nouvelle-Aquitaine',    '25': 'Bourgogne-Franche-Comté',    '26': 'Auvergne-Rhône-Alpes',    '27': 'Normandie',
    '28': 'Centre-Val de Loire',    '29': 'Bretagne',    '20': 'Corse',    '2A': 'Corse',    '2B': 'Corse',    '30': 'Occitanie',    '31': 'Occitanie',    '32': 'Occitanie',    '33': 'Nouvelle-Aquitaine',    '34': 'Occitanie',    '35': 'Bretagne',    '36': 'Centre-Val de Loire',
    '37': 'Centre-Val de Loire',    '38': 'Auvergne-Rhône-Alpes',    '39': 'Bourgogne-Franche-Comté',    '40': 'Nouvelle-Aquitaine',    '41': 'Centre-Val de Loire',    '42': 'Auvergne-Rhône-Alpes',    '43': 'Auvergne-Rhône-Alpes',    '44': 'Pays de la Loire',    
    '45': 'Centre-Val de Loire',    '46': 'Occitanie',    '47': 'Nouvelle-Aquitaine',    '48': 'Occitanie',    '49': 'Pays de la Loire',    '50': 'Normandie',    '51': 'Grand Est',    '52': 'Grand Est',    '53': 'Pays de la Loire',    '54': 'Grand Est',    '55': 'Grand Est',    '56': 'Bretagne',
    '57': 'Grand Est',    '58': 'Bourgogne-Franche-Comté',    '59': 'Hauts-de-France',    '60': 'Hauts-de-France',    '61': 'Normandie',    '62': 'Hauts-de-France',    '63': 'Auvergne-Rhône-Alpes',    '64': 'Nouvelle-Aquitaine',    '65': 'Occitanie',    '66': 'Occitanie',
    '67': 'Grand Est',    '68': 'Grand Est',    '69': 'Auvergne-Rhône-Alpes',    '70': 'Bourgogne-Franche-Comté',    '71': 'Bourgogne-Franche-Comté',    '72': 'Pays de la Loire',    '73': 'Auvergne-Rhône-Alpes',
    '74': 'Auvergne-Rhône-Alpes',    '75': 'Île-de-France',    '76': 'Normandie',    '77': 'Île-de-France',    '78': 'Île-de-France',    '79': 'Nouvelle-Aquitaine',    '80': 'Hauts-de-France',    '81': 'Occitanie',    '82': 'Occitanie',    '83': 'Provence-Alpes-Côte d\'Azur',
    '84': 'Provence-Alpes-Côte d\'Azur',    '85': 'Pays de la Loire',    '86': 'Nouvelle-Aquitaine',    '87': 'Nouvelle-Aquitaine',    '88': 'Grand Est',    '89': 'Bourgogne-Franche-Comté',    '90': 'Bourgogne-Franche-Comté',    '91': 'Île-de-France',    
    '92': 'Île-de-France',    '93': 'Île-de-France',    '94': 'Île-de-France',    '95': 'Île-de-France',    '97': 'Outre-Mer',    '98': 'Outre-Mer'
}

df['dep'] = df['dep'].astype(str)  
df['dep'] = df['dep'].apply(lambda x: x[:2] if len(x) >= 3 else x)  #On garde seulement les 2 premiers digits

df['dep']=df['dep'].replace(departement_to_region)

df = pd.concat([df.drop(columns='dep'), pd.get_dummies(df['dep'],prefix='region')], axis=1)

df_temp6=df
print(df.isna().sum().sum())
df

3552247
3548459
3546543
3545843
3545385
3545128
3544947
3544945


,Num_Acc,grav,num_veh,lum,agg,gps,lat,long,age,Casque,...,region_Corse,region_Grand Est,region_Hauts-de-France,region_Normandie,region_Nouvelle-Aquitaine,region_Occitanie,region_Outre-Mer,region_Pays de la Loire,region_Provence-Alpes-Côte d'Azur,region_Île-de-France
0,200500000001,4.0,A01,True,True,M,5051500.0,294400.0,29.0,False,...,False,False,True,False,False,False,False,False,False,False
1,200500000001,3.0,B02,True,True,M,5051500.0,294400.0,37.0,False,...,False,False,True,False,False,False,False,False,False,False
2,200500000001,1.0,B02,True,True,M,5051500.0,294400.0,41.0,False,...,False,False,True,False,False,False,False,False,False,False
3,200500000001,1.0,B02,True,True,M,5051500.0,294400.0,1.0,False,...,False,False,True,False,False,False,False,False,False,False
4,200500000001,1.0,B02,True,True,M,5051500.0,294400.0,7.0,False,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,4.0,A01,True,False,NaN,"44,9112100000","5,0196360000",46.0,False,...,False,False,False,False,False,False,False,False,False,False
2509722,202100056516,4.0,B01,True,False,NaN,"44,9112100000","5,0196360000",19.0,False,...,False,False,False,False,False,False,False,False,False,False
2509723,202100056517,3.0,A01,False,False,NaN,"44,9542747363","-0,5179211363",22.0,False,...,False,False,False,False,True,False,False,False,False,False
2509724,202100056518,3.0,A01,True,False,NaN,"48,7966700000","2,0505000000",53.0,False,...,False,False,False,False,False,False,False,False,False,True


In [12]:
"""Choix très important : Veut-on travailler sur les coordoonées ?
Soit on retire les colonne gps, lat et long                                     (choix = 1)
Soit on prend SEULEMENT les lignes ayant des coordonnées gps                    (choix = autre chose)
"""
choix = 1

df=df_temp6.copy(deep=True)
print(df.isna().sum().sum())

if choix==1 :
    df=df.drop(columns=['gps','lat','long'])
else :
    df=df[~df['gps'].isna()]  
    df=df[~df['lat'].isna()]  
    df=df[~df['long'].isna()]            

print(df.isna().sum().sum())
#Dans tous les cas, on drop.na (on en a quasiment plus)
df=df.dropna()

df_temp7=df
df

3544945
66


,Num_Acc,grav,num_veh,lum,agg,age,Casque,Ceinture,Dispositif_Enfant,Equipement_Réfléchissant,...,region_Corse,region_Grand Est,region_Hauts-de-France,region_Normandie,region_Nouvelle-Aquitaine,region_Occitanie,region_Outre-Mer,region_Pays de la Loire,region_Provence-Alpes-Côte d'Azur,region_Île-de-France
0,200500000001,4.0,A01,True,True,29.0,False,True,False,False,...,False,False,True,False,False,False,False,False,False,False
1,200500000001,3.0,B02,True,True,37.0,False,True,False,False,...,False,False,True,False,False,False,False,False,False,False
2,200500000001,1.0,B02,True,True,41.0,False,True,False,False,...,False,False,True,False,False,False,False,False,False,False
3,200500000001,1.0,B02,True,True,1.0,False,False,True,False,...,False,False,True,False,False,False,False,False,False,False
4,200500000001,1.0,B02,True,True,7.0,False,True,False,False,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,4.0,A01,True,False,46.0,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2509722,202100056516,4.0,B01,True,False,19.0,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2509723,202100056517,3.0,A01,False,False,22.0,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2509724,202100056518,3.0,A01,True,False,53.0,False,True,False,False,...,False,False,False,False,False,False,False,False,False,True


In [13]:
#sauvegarde du csv pre-processed
df.to_csv('001_df_processed.csv', index=False)